In [ ]:
import sys
import os
# Support: repo root, parent of repo, or run from notebooks/ (e.g. jupyter nbconvert --execute)
_REPO_NAME = 'Physically-conditioned-latent-diffusion-model-for-temperature'
_cwd = os.getcwd()
if os.path.basename(_cwd) == _REPO_NAME:
    pass  # already at repo root
elif os.path.isdir(os.path.join(_cwd, _REPO_NAME)):
    os.chdir(os.path.join(_cwd, _REPO_NAME))
elif os.path.basename(_cwd) == 'notebooks' and os.path.basename(os.path.dirname(_cwd)) == _REPO_NAME:
    os.chdir(os.path.dirname(_cwd))  # run from notebooks/ -> go to repo root
else:
    pass
sys.path.insert(0, os.getcwd())
import torch
from tqdm import tqdm
import pickle 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import lightning as L
seed = 150
L.seed_everything(seed, workers=True)

from src.models.unet_module import UnetLitModule
from src.models.gan_module import UnetGANLitModule
from src.models.ae_module import AutoencoderKL, EncoderLRES
from src.models.ldm_module import LatentDiffusion
from src.models.lmm_module import LatentMeanFlowLitModule
from src.models.components.meanflow.meanflow_core import MeanFlowCore

from src.models.components.unet import DownscalingUnet
from src.models.components.ae import SimpleConvDecoder, SimpleConvEncoder
from src.models.components.ldm.denoiser import UNetModel, DDIMSampler, MFUNet
from src.models.components.ldm.conditioner import AFNOConditionerNetCascade
from src.data.downscaling_datamodule import DownscalingDataModule
from src.data.components.downscaling_dataset import DownscalingDataset 

from utils.inference_utils import get_model_output
from utils.plotting_utils import get_target_grid, from_torchtensor_to_xarray, quadratic_regrid_era5_xr_to_high_res, linear_regrid_era5_xr_to_high_res 


In [ ]:

# Set paths — dataset on project mount: ``LDM-downscaling/full_Dataset/`` (same as training ``paths.data_dir``).
# Slurm ``Submit.sh`` sets ``LDM_DATA_ROOT`` to an absolute path; override locally if your mount differs.
trained_models_path = './pretrained_models/'
_d = os.environ.get('LDM_DATA_ROOT', '').strip()
data_path = _d if _d else './LDM-downscaling/full_Dataset/'
if not data_path.endswith(os.sep):
    data_path = data_path + os.sep
output_path = './outputs/'


static_vars = {'dtm_tif_file': data_path + 'static_var/dtm_2km_domain_trim_EPSG3035.tif',
               'lc_tif_file': data_path + 'static_var/land_cover_classes_2km_domain_trim_EPSG3035.tif',
               'lat_tif_file': data_path + 'static_var/lat_2km_domain_trim_EPSG3035.tif'}

borders_file = data_path + 'plotting_resources/borders_downscaling_domain_3035.geojson'

# Read in the data normalization info
with open(data_path + 'normalization_data.pkl', 'rb') as f:
    norm_values = pickle.load(f)

# Max timesteps to run (capped by actual test loader length after ``attach_inference_test_loader``).
nr_timesteps = 165

# If True: use Lightning ``DownscalingDataModule.setup`` 15% test split (same seed as training, ``metadata.csv``).
# If False: use ``metadata_test_paper_sample.csv`` (paper subset; file must exist under ``data_path``).
USE_SPLIT_TEST_FOR_INFERENCE = True


def attach_inference_test_loader(data_module, use_split: bool, *, data_path, target_vars, nn_lowres, static_vars):
    if use_split:
        data_module.setup('test')
    else:
        data_module.data_test = DownscalingDataset(
            data_path,
            target_vars=target_vars,
            nn_lowres=nn_lowres,
            static_vars=static_vars,
            metadata_file_name='metadata_test_paper_sample.csv',
        )


def inference_timestep_cap(data_module, cap: int) -> int:
    return min(cap, len(data_module.data_test))

In [4]:
# Choose the target variables (either '2mT' or 'UV')
target_var = '2mT'
#target_var = 'UV'

target_channels = 1 if target_var == '2mT' else 2
target_vars = {'high_res': ['2mT']} if target_var == '2mT' else {'high_res': ['U10', 'V10']}
target_vars['low_res'] = ['2mT', 'PMSL', 'U10', 'V10', 'dp2mT', 'SST', 'SNDPT', 'TP', 'SSRadIn', 'Q850', 'T850', 'U850', 'V850', 'W850']

tv_idxs = {}
for tv in target_vars['high_res']:
    tv_idxs[tv] = [i for i in range(len(target_vars['low_res'])) if target_vars['low_res'][i] == tv][0]

In [ ]:
# Load the UNET model 
ckpt_ref_file = trained_models_path + 'UNET_' + target_var + '.ckpt'
loss = torch.nn.Identity()
in_ch = 18 + 14  # 18 static variables + 14 predictors
out_ch = target_channels
unet_model = UnetLitModule.load_from_checkpoint(ckpt_ref_file,
                                           net=DownscalingUnet(in_ch=in_ch,out_ch=out_ch),
                                           loss=loss,
                                           ckpt_path=None,
                                           strict=False).eval().to(device='cuda:0') 

nn_lowres = True
data_module_unet = DownscalingDataModule(
    data_dir=data_path,
    target_vars=target_vars,
    batch_size=1,
    num_workers=1,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
    metadata_file_name='metadata.csv',
)
attach_inference_test_loader(
    data_module_unet,
    USE_SPLIT_TEST_FOR_INFERENCE,
    data_path=data_path,
    target_vars=target_vars,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
)
nr_timesteps = inference_timestep_cap(data_module_unet, nr_timesteps)
test_dataloader_unet = data_module_unet.test_dataloader()
test_iter_unet = iter(test_dataloader_unet)
unet_preds_norm = {}
# Run the UNET on the test dataset
for idx_preds in tqdm(range(0, nr_timesteps)):           
    el = next(test_iter_unet)
    model_output, ts_ns = get_model_output('unet-like', unet_model, el)
    ts_ns = pd.to_datetime(int(ts_ns))
    unet_preds_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        unet_preds_norm[ts_ns][tv] = model_output.cpu()[0,index,:,:]* norm_values['CMCC']['std'][tv] + norm_values['CMCC']['mean'][tv]
    if target_var == 'UV':
        unet_preds_norm[ts_ns]['WS10'] = np.sqrt(unet_preds_norm[ts_ns]['U10'] **2 + unet_preds_norm[ts_ns]['V10']**2)

# Plot data
plot_var = '2mT' if target_var=='2mT' else 'WS10'
fig, ax_unet = plt.subplots(ncols=nr_timesteps,figsize=(4*nr_timesteps,5),sharey=True, sharex=True, constrained_layout=True)
ax_unet = np.atleast_1d(ax_unet)
fig.suptitle('UNET')
for idx_preds,ts_ns in enumerate(unet_preds_norm):
    ax_unet[idx_preds].imshow(unet_preds_norm[ts_ns][plot_var])
    ax_unet[idx_preds].set_title(ts_ns, fontsize=10) 

In [ ]:
# Load the GAN model
ckpt_ref_file = trained_models_path + 'GAN_' + target_var + '.ckpt'
loss = torch.nn.Identity()
in_ch = 18 + 14  # 18 static variables + 14 predictors
out_ch = target_channels
gan_model = UnetGANLitModule.load_from_checkpoint(ckpt_ref_file,
                                              net=DownscalingUnet(in_ch=in_ch,out_ch=out_ch),
                                              loss=loss,
                                              ckpt_path=None,
                                              strict=False).eval().to(device='cuda:0')
nn_lowres = True
data_module_gan = DownscalingDataModule(
    data_dir=data_path,
    target_vars=target_vars,
    batch_size=1,
    num_workers=1,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
    metadata_file_name='metadata.csv',
)
attach_inference_test_loader(
    data_module_gan,
    USE_SPLIT_TEST_FOR_INFERENCE,
    data_path=data_path,
    target_vars=target_vars,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
)
test_dataloader_gan = data_module_gan.test_dataloader()
test_iter_gan = iter(test_dataloader_gan)
gan_preds_norm = {}
# Run the GAN on the test dataset
for idx_preds in tqdm(range(0, nr_timesteps)):           
    el = next(test_iter_gan)
    model_output, ts_ns = get_model_output('unet-like', gan_model, el)
    ts_ns = pd.to_datetime(int(ts_ns))
    gan_preds_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        gan_preds_norm[ts_ns][tv] = model_output.cpu()[0,index,:,:]* norm_values['CMCC']['std'][tv] + norm_values['CMCC']['mean'][tv]
    if target_var == 'UV':
        gan_preds_norm[ts_ns]['WS10'] = np.sqrt(gan_preds_norm[ts_ns]['U10'] **2 + gan_preds_norm[ts_ns]['V10']**2)

# Plot data
plot_var = '2mT' if target_var=='2mT' else 'WS10'
fig, ax_gan = plt.subplots(ncols=nr_timesteps,
                           figsize=(4*nr_timesteps, 5),
                           sharey=True, sharex=True,
                           constrained_layout=True)
ax_gan = np.atleast_1d(ax_gan)       

fig.suptitle('GAN')
for idx_preds,ts_ns in enumerate(gan_preds_norm):
    ax_gan[idx_preds].imshow(gan_preds_norm[ts_ns][plot_var])
    ax_gan[idx_preds].set_title(ts_ns, fontsize=10) 




In [ ]:


# File paths 
ckpt_ref_file = trained_models_path + 'LDM_residual_' + target_var + '.ckpt'
ckpt_partial_file = trained_models_path + "pde_loss_model_checkpoint.ckpt"
ae_ckpt_ref_file = trained_models_path + 'VAE_residual_' + target_var + '.ckpt'

# Model hyper-parameters 
in_dim = target_channels
parameterization = 'v'
ae_flag = 'residual'
unet_regr = unet_model  # assume unet_model is defined

# Combine Checkpoints
# Load the full (reference) checkpoint state dict.
full_ckpt = torch.load(ckpt_ref_file, map_location='cuda:0', weights_only=False)
# Load the partial checkpoint state dict (the one with frozen parameters).
partial_ckpt = torch.load(ckpt_partial_file, map_location='cuda:0', weights_only=False)

# Get the state dictionaries from each checkpoint.
state_full = full_ckpt["state_dict"]
state_partial = partial_ckpt["state_dict"]

# Combine: use the values in state_partial to override the full state.
combined_state = state_full.copy()
combined_state.update(state_partial)
# (Now any missing keys in the partial checkpoint are filled in from the full checkpoint)

# Instantiate the Model 
ldm_res_model = LatentDiffusion(
    denoiser=UNetModel(
                in_channels=32*in_dim,
                model_channels=256,
                out_channels=32*in_dim,
                num_res_blocks=2,
                attention_resolutions=[1,2],
                dims=2,
                channel_mult=[1,2,4],
                num_heads=8,
                context_ch=[256,512,1024]),
    autoencoder=AutoencoderKL(
                encoder=SimpleConvEncoder(in_dim=in_dim, levels=3),
                decoder=SimpleConvDecoder(in_dim=in_dim, levels=3),
                ae_flag=ae_flag, 
                unet_regr=unet_regr),
    context_encoder=AFNOConditionerNetCascade(
                autoencoder=[AutoencoderKL(encoder=SimpleConvEncoder(in_dim=18, levels=3, ch_mult=3),
                                            decoder=None),
                             EncoderLRES()],
                train_autoenc=False,
                cascade_depth=3,
                embed_dim=[128,24],
                analysis_depth=[4,4],
                afno_fusion=True,
                input_size_ratios=[1,1],
                embed_dim_out=256),
    ae_load_state_file=ae_ckpt_ref_file, 
    parameterization=parameterization
).to(device='cuda:0')

# Load the Combined State Dictionary
# Use strict=False so that any (noncritical) missing keys are ignored.
ldm_res_model.load_state_dict(combined_state, strict=False)
ldm_res_model.eval()

# Set Up the Test DataLoader and Sampler
nn_lowres = False
data_module_ldm = DownscalingDataModule(
    data_dir=data_path,
    target_vars=target_vars,
    batch_size=1,
    num_workers=1,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
    metadata_file_name='metadata.csv',
)
attach_inference_test_loader(
    data_module_ldm,
    USE_SPLIT_TEST_FOR_INFERENCE,
    data_path=data_path,
    target_vars=target_vars,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
)
test_dataloader_ldm = data_module_ldm.test_dataloader()
test_iter_ldm = iter(test_dataloader_ldm)

sampler = DDIMSampler(ldm_res_model)
denoising_steps = 100

# Run Inference on the Test Dataset
ldm_preds_norm = {}
for idx_preds in tqdm(range(0, nr_timesteps)):
    el = next(test_iter_ldm)
    model_output, ts_ns = get_model_output('ldm', ldm_res_model, el, sampler=sampler, num_diffusion_iters=denoising_steps)
    ts_ns = pd.to_datetime(int(ts_ns))
    ldm_preds_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        ldm_preds_norm[ts_ns][tv] = model_output.cpu()[0, index, :, :] * norm_values['CMCC']['std'][tv] + norm_values['CMCC']['mean'][tv]
    if target_var == 'UV':
        ldm_preds_norm[ts_ns]['WS10'] = np.sqrt(ldm_preds_norm[ts_ns]['U10']**2 + ldm_preds_norm[ts_ns]['V10']**2)

# plot
plot_var = '2mT' if target_var == '2mT' else 'WS10'
fig, ax_ldm = plt.subplots(ncols=nr_timesteps, figsize=(4*nr_timesteps, 5), sharey=True, sharex=True, constrained_layout=True)
fig.suptitle('LDM_res With Our PDE')
for idx_preds, ts_ns in enumerate(ldm_preds_norm):
    ax_ldm[idx_preds].imshow(ldm_preds_norm[ts_ns][plot_var])
    ax_ldm[idx_preds].set_title(str(ts_ns), fontsize=10)
plt.show()
#Save for later more plotting
ldm_preds_pde = ldm_preds_norm.copy()


In [ ]:
# Latent MeanFlow (LMM): align with cluster training — `configs/experiment/Submitscript.sh` runs
# `train.py experiment=downscaling_LMM_res_2mT` (see `configs/experiment/downscaling_LMM_res_2mT.yaml`
# + `configs/model/lmm.yaml`). For 2mT we build `cfg.model` with Hydra so the graph matches training.
#
# MeanFlow note (`lmm_module.py` ~68–72): when `use_meanflow_paper_core=True`, the active object is
# `MeanFlowPaperCore(**(meanflow_paper or {}))`; Hydra's `meanflow_core` (MeanFlowCore) is still
# constructed but not used as `self.meanflow_core` — do not tune MeanFlowCore kwargs for this run.
#
# Optional `lmm_pde_loss_model_checkpoint.ckpt`: merge extra LMM weights (`mf_unet.*`, etc.), not LDM denoiser keys.
# Main LMM weights: Lightning best checkpoint from training (not under ``pretrained_models/``).
if target_var == '2mT':
    ckpt_lmm_ref = './logs/train/runs/2026-04-24_18-27-35/checkpoints/best_LMM_res_2mT_epoch_032.ckpt'
else:
    ckpt_lmm_ref = trained_models_path + 'LMM_residual_' + target_var + '.ckpt'
ckpt_lmm_pde = trained_models_path + 'lmm_pde_loss_model_checkpoint.ckpt'

os.environ.setdefault('PROJECT_ROOT', os.getcwd())  # same as `paths/default.yaml` for pretrained paths

if target_var == '2mT':
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate

    _config_dir = os.path.join(os.getcwd(), 'configs')
    with initialize_config_dir(version_base=None, config_dir=_config_dir):
        cfg = compose(config_name='train', overrides=['experiment=downscaling_LMM_res_2mT'])
    lmm_model = instantiate(cfg.model).to(device='cuda:0')
else:
    # No `downscaling_LMM_res_*` experiment for UV in-repo — mirror `configs/model/lmm.yaml` manually.
    in_dim = target_channels
    ae_ckpt_ref_file = trained_models_path + 'VAE_residual_' + target_var + '.ckpt'
    meanflow_core = MeanFlowCore()  # unused when use_meanflow_paper_core=True (see note above)

    lmm_model = LatentMeanFlowLitModule(
        mf_unet=MFUNet(
            in_channels=32 * in_dim,
            model_channels=256,
            out_channels=32 * in_dim,
            num_res_blocks=2,
            attention_resolutions=[1, 2],
            dims=2,
            channel_mult=[1, 2, 4],
            num_heads=8,
            context_ch=[256, 512, 1024],
            time_scale=1000.0,
            separate_r_mlp=False,
        ),
        meanflow_core=meanflow_core,
        autoencoder=AutoencoderKL(
            encoder=SimpleConvEncoder(in_dim=in_dim, levels=3),
            decoder=SimpleConvDecoder(in_dim=in_dim, levels=3),
            ae_flag='residual',
            unet_regr=unet_model,
        ),
        context_encoder=AFNOConditionerNetCascade(
            autoencoder=[
                AutoencoderKL(
                    encoder=SimpleConvEncoder(in_dim=18, levels=3, ch_mult=3),
                    decoder=None,
                ),
                EncoderLRES(),
            ],
            train_autoenc=True,
            cascade_depth=3,
            embed_dim=[128, 24],
            analysis_depth=[4, 4],
            afno_fusion=True,
            input_size_ratios=[1, 1],
            embed_dim_out=256,
        ),
        ae_load_state_file=ae_ckpt_ref_file,
        use_meanflow_paper_core=True,
        meanflow_paper=None,
    ).to(device='cuda:0')

state_lmm = torch.load(ckpt_lmm_ref, map_location='cuda:0', weights_only=False)['state_dict']
if os.path.isfile(ckpt_lmm_pde):
    state_partial_lmm = torch.load(ckpt_lmm_pde, map_location='cuda:0', weights_only=False)['state_dict']
    merged_lmm = state_lmm.copy()
    merged_lmm.update(state_partial_lmm)
    state_lmm = merged_lmm
lmm_model.load_state_dict(state_lmm, strict=False)
lmm_model.eval()

nn_lowres = False
data_module_lmm = DownscalingDataModule(
    data_dir=data_path,
    target_vars=target_vars,
    batch_size=1,
    num_workers=1,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
    metadata_file_name='metadata.csv',
)
attach_inference_test_loader(
    data_module_lmm,
    USE_SPLIT_TEST_FOR_INFERENCE,
    data_path=data_path,
    target_vars=target_vars,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
)
test_dataloader_lmm = data_module_lmm.test_dataloader()
test_iter_lmm = iter(test_dataloader_lmm)

lmm_preds_norm = {}
for idx_preds in tqdm(range(0, nr_timesteps)):
    el = next(test_iter_lmm)
    model_output, ts_ns = get_model_output('lmm', lmm_model, el)
    ts_ns = pd.to_datetime(int(ts_ns))
    lmm_preds_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        lmm_preds_norm[ts_ns][tv] = (
            model_output.cpu()[0, index, :, :]
            * norm_values['CMCC']['std'][tv]
            + norm_values['CMCC']['mean'][tv]
        )
    if target_var == 'UV':
        lmm_preds_norm[ts_ns]['WS10'] = np.sqrt(
            lmm_preds_norm[ts_ns]['U10'] ** 2 + lmm_preds_norm[ts_ns]['V10'] ** 2
        )

plot_var = '2mT' if target_var == '2mT' else 'WS10'
fig, ax_lmm = plt.subplots(
    ncols=nr_timesteps,
    figsize=(4 * nr_timesteps, 5),
    sharey=True,
    sharex=True,
    constrained_layout=True,
)
fig.suptitle('LMM (best_LMM_res_2mT_epoch_032); optional merge from lmm_pde ckpt')
for idx_preds, ts_ns in enumerate(lmm_preds_norm):
    ax_lmm[idx_preds].imshow(lmm_preds_norm[ts_ns][plot_var])
    ax_lmm[idx_preds].set_title(str(ts_ns), fontsize=10)
plt.show()
lmm_preds_pde = lmm_preds_norm.copy()


In [8]:
# Load the LDM_res model
ckpt_ref_file = trained_models_path + 'LDM_residual_' + target_var + '.ckpt'
ae_ckpt_ref_file = trained_models_path + 'VAE_residual_' + target_var + '.ckpt'
in_dim = target_channels
parameterization = 'v'
ae_flag = 'residual'
unet_regr = unet_model
ldm_res_model = LatentDiffusion.load_from_checkpoint(ckpt_ref_file,
                                            denoiser=UNetModel(in_channels=32*in_dim,
                                                            model_channels=256,
                                                            out_channels=32*in_dim,
                                                            num_res_blocks=2,
                                                            attention_resolutions=[1,2],
                                                            dims=2,
                                                            channel_mult=[1,2,4],
                                                            num_heads=8,
                                                            context_ch=[256,512,1024]),
                                            autoencoder=AutoencoderKL(encoder=SimpleConvEncoder(in_dim=in_dim, levels=3),
                                                                      decoder=SimpleConvDecoder(in_dim=in_dim, levels=3),
                                                                      ae_flag=ae_flag, 
                                                                      unet_regr=unet_regr),
                                            context_encoder=AFNOConditionerNetCascade(autoencoder=[AutoencoderKL(encoder=SimpleConvEncoder(in_dim=18,levels=3,ch_mult=3),
                                                                                                             decoder=None),
                                                                                               EncoderLRES()], 
                                                                                  train_autoenc=False,
                                                                                  cascade_depth=3,
                                                                                  embed_dim=[128,24],
                                                                                  analysis_depth=[4,4],
                                                                                  afno_fusion=True,
                                                                                  input_size_ratios=[1,1],
                                                                                  embed_dim_out=256),
                                            ae_load_state_file=ae_ckpt_ref_file, 
                                            parameterization=parameterization).eval().to(device='cuda:0')
nn_lowres = False
data_module_ldm = DownscalingDataModule(
    data_dir=data_path,
    target_vars=target_vars,
    batch_size=1,
    num_workers=1,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
    metadata_file_name='metadata.csv',
)
attach_inference_test_loader(
    data_module_ldm,
    USE_SPLIT_TEST_FOR_INFERENCE,
    data_path=data_path,
    target_vars=target_vars,
    nn_lowres=nn_lowres,
    static_vars=static_vars,
)
test_dataloader_ldm = data_module_ldm.test_dataloader()
test_iter_ldm = iter(test_dataloader_ldm)
# Set up sampler
sampler = DDIMSampler(ldm_res_model)
denoising_steps = 100

ldm_preds_norm = {}
# Run the LDM on the test dataset
for idx_preds in tqdm(range(0,nr_timesteps)):          
    el = next(test_iter_ldm)
    model_output, ts_ns = get_model_output('ldm', ldm_res_model, el, sampler=sampler, num_diffusion_iters=denoising_steps)
    ts_ns = pd.to_datetime(int(ts_ns))
    ldm_preds_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        ldm_preds_norm[ts_ns][tv] = model_output.cpu()[0,index,:,:]* norm_values['CMCC']['std'][tv] + norm_values['CMCC']['mean'][tv]
    if target_var == 'UV':
        ldm_preds_norm[ts_ns]['WS10'] = np.sqrt(ldm_preds_norm[ts_ns]['U10'] **2 + ldm_preds_norm[ts_ns]['V10']**2)

# Plot data
plot_var = '2mT' if target_var=='2mT' else 'WS10'
fig, ax_ldm = plt.subplots(ncols=nr_timesteps,figsize=(4*nr_timesteps,5),sharey=True, sharex=True, constrained_layout=True)
fig.suptitle('LDM_res Original Model')
for idx_preds,ts_ns in enumerate(ldm_preds_norm):
    ax_ldm[idx_preds].imshow(ldm_preds_norm[ts_ns][plot_var])
    ax_ldm[idx_preds].set_title(ts_ns, fontsize=10) 



In [9]:
# Store ERA5 and COSMO-CLM baseline datasets
test_iter_unet = iter(test_dataloader_unet)
era5_norm = {}
cosmo_norm = {}

for idx_preds in tqdm(range(0,nr_timesteps)):
    el = next(test_iter_unet)
    low_res = el[0]
    high_res = el[1]
    ts_ns = pd.to_datetime(int(el[-1]))  # ref_time last: (low, high, time) or (low, high, static, time)
    era5_norm[ts_ns] = {}
    cosmo_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        era5_norm[ts_ns][tv] = low_res[0, tv_idxs[tv],:,:]* norm_values['ERA5']['std'][tv] + norm_values['ERA5']['mean'][tv]
        cosmo_norm[ts_ns][tv] = high_res[0,index,:,:]* norm_values['CMCC']['std'][tv] + norm_values['CMCC']['mean'][tv]
    if target_var == 'UV':
        era5_norm[ts_ns]['WS10'] = np.sqrt(era5_norm[ts_ns]['U10'] **2 + era5_norm[ts_ns]['V10']**2)
        cosmo_norm[ts_ns]['WS10'] = np.sqrt(cosmo_norm[ts_ns]['U10'] **2 + cosmo_norm[ts_ns]['V10']**2)

# Store Quadratic + Linear interp baselines (same ERA5 low-res; compare metrics in metric_computation).
test_iter_ldm = iter(test_dataloader_ldm)
quadratic_norm = {}
linear_norm = {}

target_grid_low_res = get_target_grid('low')
target_grid_high_res = get_target_grid('high')

for idx_preds in tqdm(range(0,nr_timesteps)):
    el = next(test_iter_ldm)
    low_res = el[0]
    ts_ns = pd.to_datetime(int(el[-1]))
    quadratic_norm[ts_ns] = {}
    linear_norm[ts_ns] = {}
    for index, tv in enumerate(target_vars['high_res']):
        # De-apply normalization
        era5_xr = from_torchtensor_to_xarray(low_res[0, tv_idxs[tv],:,:], target_grid_low_res, coords_name='y_x')
        quadratic_interp = quadratic_regrid_era5_xr_to_high_res(era5_xr, target_grid_high_res)
        quadratic_interp = torch.from_numpy(quadratic_interp.values)
        quadratic_norm[ts_ns][tv] = quadratic_interp* norm_values['ERA5']['std'][tv] + norm_values['ERA5']['mean'][tv]
        linear_interp = linear_regrid_era5_xr_to_high_res(era5_xr, target_grid_high_res)
        linear_interp = torch.from_numpy(linear_interp.values)
        linear_norm[ts_ns][tv] = linear_interp* norm_values['ERA5']['std'][tv] + norm_values['ERA5']['mean'][tv]
    if target_var == 'UV':
        quadratic_norm[ts_ns]['WS10'] = np.sqrt(quadratic_norm[ts_ns]['U10'] **2 + quadratic_norm[ts_ns]['V10']**2)
        linear_norm[ts_ns]['WS10'] = np.sqrt(linear_norm[ts_ns]['U10'] **2 + linear_norm[ts_ns]['V10']**2)

# Plot data
mod_list = {'ERA5': era5_norm, 'COSMO-CLM': cosmo_norm, 'Quadratic Interp.': quadratic_norm, 'Linear Interp.': linear_norm}
for mod_i in mod_list:
    fig, ax = plt.subplots(ncols=nr_timesteps,figsize=(4*nr_timesteps,5),sharey=True, sharex=True, constrained_layout=True)
    fig.suptitle(mod_i)
    for idx_preds,ts_ns in enumerate(quadratic_norm):
        ax[idx_preds].imshow(mod_list[mod_i][ts_ns][plot_var])
        ax[idx_preds].set_title(ts_ns, fontsize=10) 


In [10]:
# Prepare data for saving results and later plotting
results = {'ERA5': era5_norm,
           'COSMO-CLM': cosmo_norm,
           'Quadratic Interp.': quadratic_norm,
           'Linear Interp.': linear_norm,
           'UNET': unet_preds_norm,
           'GAN': gan_preds_norm,
           'LDM_res': ldm_preds_norm,
           'LDM_PDE_res': ldm_preds_pde,
           'LMM_PDE_res': lmm_preds_pde}

d = []
for mod_i in results:
    for ts in results[mod_i]:
        pred_ts = results[mod_i][ts]
        # print(pred_ts.dims)
        for tv in target_vars['high_res']:
            d.append({'input_var': 'all',
                      'target_var': target_var,
                      'model': mod_i,
                      'variable': tv,
                      'spat_distr': pred_ts[tv],
                      'min': np.percentile(pred_ts[tv],0.5),
                      'max': np.percentile(pred_ts[tv],99.5),
                      'time_step': ts})
        if target_var == 'UV':
            d.append({'input_var': 'all',
                      'target_var': target_var,
                      'model': mod_i,
                      'variable': 'WS10',
                      'spat_distr': pred_ts['WS10'],
                      'min': np.percentile(pred_ts['WS10'],0.5),
                      'max': np.percentile(pred_ts['WS10'],99.5),
                      'time_step': ts})
results_df = pd.DataFrame(d)

In [11]:
# Save results for later plotting
results_df.to_pickle(output_path + './Our_results_trained_models_' + target_var + '.pkl')  

In [12]:
 # Load the VAE model
#ckpt_ref_file = trained_models_path + 'VAE_' + target_var + '.ckpt'
#in_dim = target_channels
#ae_flag = 'residual'
#unet_regr = unet_model
#vae_model = AutoencoderKL.load_from_checkpoint(ckpt_ref_file,
#                                         encoder=SimpleConvEncoder(in_dim=in_dim, levels=3),
#                                         decoder=SimpleConvDecoder(in_dim=in_dim, levels=3),
#                                         ae_flag=ae_flag,
#                                         unet_regr = unet_regr).eval().to(device='cuda:0')      

In [13]:
# %% Plot Outputs in Three Rows: PDE-based LDM, Original LDM, and COSMO-CLM Baseline

import matplotlib.pyplot as plt

# Get sorted timestamps to display time steps in order.
timestamps = sorted(list(cosmo_norm.keys()))

# Create a figure with 3 rows (one per model) and columns equal to the number of time steps.
fig, axes = plt.subplots(nrows=3, ncols=nr_timesteps, figsize=(4*nr_timesteps, 15),
                         sharex=True, sharey=True, constrained_layout=True)

# First row: Our PDE-based LDM outputs
for col, ts in enumerate(timestamps):
    axes[0, col].imshow(ldm_preds_pde[ts][plot_var])
    axes[0, col].set_title(f"PDE LDM\n{ts}", fontsize=10)
    axes[0, col].axis('off')
axes[0, 0].set_ylabel("PDE-based LDM", fontsize=12, rotation=90, labelpad=10)

# Second row: Original LDM outputs (stored in ldm_preds_norm, renamed to ldm_preds_Original for clarity)
for col, ts in enumerate(timestamps):
    axes[1, col].imshow(ldm_preds_norm[ts][plot_var])
    axes[1, col].set_title(f"Original LDM\n{ts}", fontsize=10)
    axes[1, col].axis('off')
axes[1, 0].set_ylabel("Original LDM", fontsize=12, rotation=90, labelpad=10)

# Third row: COSMO-CLM baseline outputs
for col, ts in enumerate(timestamps):
    axes[2, col].imshow(cosmo_norm[ts][plot_var])
    axes[2, col].set_title(f"COSMO-CLM\n{ts}", fontsize=10)
    axes[2, col].axis('off')
axes[2, 0].set_ylabel("COSMO-CLM", fontsize=12, rotation=90, labelpad=10)

fig.suptitle("Comparison of Outputs: PDE-based LDM vs Original LDM vs COSMO-CLM", fontsize=16)
plt.show()


In [ ]:
# %% Export animations (no cropping): prediction (top) vs COSMO-CLM (bottom)
# This cell assumes these dicts exist from earlier cells:
# - unet_preds_norm, gan_preds_norm, ldm_preds_pde (PDE-based), ldm_preds_norm (original), cosmo_norm
# And that plot_var is set ('2mT' or 'WS10').

from pathlib import Path
import numpy as np

out_anim_dir = Path(output_path) / "animations" / target_var
out_anim_dir.mkdir(parents=True, exist_ok=True)

# Try to enable MP4 export (requires ffmpeg). GIF export will work without ffmpeg.
try:
    import imageio.v2 as imageio  # pip install imageio
except Exception as e:
    raise RuntimeError(
        "Missing dependency 'imageio'. Install with: pip install imageio"
    ) from e

# If ffmpeg isn't available, imageio can still write GIF; MP4 will be skipped.
try:
    import imageio_ffmpeg  # noqa: F401  (pip install imageio-ffmpeg)
    _HAVE_FFMPEG = True
except Exception:
    _HAVE_FFMPEG = False


def _global_vmin_vmax(pred_dict, truth_dict, ts_list, var):
    vals = []
    for ts in ts_list:
        a = np.asarray(pred_dict[ts][var])
        b = np.asarray(truth_dict[ts][var])
        vals.append(a)
        vals.append(b)
    stacked = np.stack(vals, axis=0)
    vmin = float(np.nanpercentile(stacked, 0.5))
    vmax = float(np.nanpercentile(stacked, 99.5))
    return vmin, vmax


def export_pred_vs_truth(pred_name, pred_dict, truth_dict, ts_list, var, *, fps=6):
    vmin, vmax = _global_vmin_vmax(pred_dict, truth_dict, ts_list, var)

    gif_path = out_anim_dir / f"{pred_name}_vs_COSMO-CLM_{var}.gif"
    mp4_path = out_anim_dir / f"{pred_name}_vs_COSMO-CLM_{var}.mp4"

    gif_writer = imageio.get_writer(gif_path, mode="I", duration=1 / fps)
    mp4_writer = None
    if _HAVE_FFMPEG:
        # macro_block_size=None avoids resizing to multiples of 16
        mp4_writer = imageio.get_writer(
            mp4_path, fps=fps, codec="libx264", quality=8, macro_block_size=None
        )

    try:
        import matplotlib.pyplot as plt
        from matplotlib import colormaps as cmaps

        cmap = cmaps.get_cmap("viridis")

        def to_rgb(x):
            x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
            rgb = (cmap(x)[..., :3] * 255).astype(np.uint8)
            return rgb

        for ts in ts_list:
            pred = np.asarray(pred_dict[ts][var])
            truth = np.asarray(truth_dict[ts][var])

            # Add simple titles by rendering once with matplotlib (fast enough for ~100–200 frames)
            fig, axes = plt.subplots(
                nrows=2, ncols=1, figsize=(6, 10), constrained_layout=True
            )
            axes[0].imshow(pred, vmin=vmin, vmax=vmax, cmap=cmap)
            axes[0].set_title(f"{pred_name} | {ts}")
            axes[0].axis("off")
            axes[1].imshow(truth, vmin=vmin, vmax=vmax, cmap=cmap)
            axes[1].set_title(f"COSMO-CLM | {ts}")
            axes[1].axis("off")

            # Draw to array (RGBA -> RGB)
            fig.canvas.draw()
            buf = fig.canvas.buffer_rgba()
            frame = np.frombuffer(buf, dtype=np.uint8)
            frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (4,))
            frame = frame[..., :3]  # drop alpha
            plt.close(fig)

            gif_writer.append_data(frame)
            if mp4_writer is not None:
                mp4_writer.append_data(frame)
    finally:
        gif_writer.close()
        if mp4_writer is not None:
            mp4_writer.close()

    print(f"Saved GIF: {gif_path}")
    if _HAVE_FFMPEG:
        print(f"Saved MP4: {mp4_path}")
    else:
        print("MP4 skipped (ffmpeg not available). Install: pip install imageio-ffmpeg")


# Choose models to export (add/remove as you like)
models_to_export = {
    "UNET": unet_preds_norm,
    "GAN": gan_preds_norm,
    "LDM_PDE": ldm_preds_pde,
    "LDM_Original": ldm_preds_norm,
    "LMM_PDE": lmm_preds_pde,
}

# Use COSMO-CLM timestamps as canonical ordering
timestamps = sorted(list(cosmo_norm.keys()))

for name, pred in models_to_export.items():
    export_pred_vs_truth(name, pred, cosmo_norm, timestamps, plot_var, fps=6)
